In [1]:
import requests
from bs4 import BeautifulSoup
import time
import sqlite3

# データベースに接続
db_name = 'github_repos.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# テーブルの初期化
cursor.execute("DROP TABLE IF EXISTS google_repos")
cursor.execute("""
    CREATE TABLE google_repos (
        repo_name TEXT,
        language TEXT,
        stars INTEGER
    )
""")
conn.commit()
print("データベースとテーブルの準備が完了しました。")

データベースとテーブルの準備が完了しました。


In [6]:

import sqlite3
import requests
from bs4 import BeautifulSoup
import time

# DB再接続
conn = sqlite3.connect('github_repos.db')
cursor = conn.cursor()

url = 'https://github.com/google?tab=repositories'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

print(f"アクセス中: {url}")
response = requests.get(url, headers=headers)
time.sleep(1)

# ★デバッグ用：取得したHTMLを保存して、あとで人間が確認できるようにする
with open("debug_github.html", "w", encoding="utf-8") as f:
    f.write(response.text)
print("-> 取得したHTMLを 'debug_github.html' に保存しました（左のファイル一覧に出ます）")

soup = BeautifulSoup(response.text, 'html.parser')


repo_links = soup.select('a[itemprop="name codeRepository"]')

print(f"{len(repo_links)} 件のリポジトリ候補が見つかりました。")

count = 0
for link in repo_links:
    try:
        # リポジトリ名
        name = link.text.strip()
        
     
        repo_block = link.find_parent('li')
        

        if not repo_block:
            repo_block = link.find_parent('div', class_='col-10')

        if repo_block:
            # 言語
            lang_tag = repo_block.find('span', itemprop='programmingLanguage')
            language = lang_tag.text.strip() if lang_tag else "Unknown"

            # スター数
            star_tag = repo_block.find('a', href=lambda x: x and '/stargazers' in x)
            if star_tag:
                star_text = star_tag.text.strip().replace(',', '')
                if 'k' in star_text.lower():
                    stars = int(float(star_text.lower().replace('k', '')) * 1000)
                else:
                    stars = int(star_text)
            else:
                stars = 0
        else:
            language = "Unknown"
            stars = 0

        # DBへ保存
        cursor.execute("INSERT OR REPLACE INTO google_repos VALUES (?, ?, ?)", (name, language, stars))
        count += 1
        
    except Exception as e:
        print(f"Error: {e}")
        continue

conn.commit()
print(f"★ {count} 件のデータを保存しました。")


アクセス中: https://github.com/google?tab=repositories
-> 取得したHTMLを 'debug_github.html' に保存しました（左のファイル一覧に出ます）
10 件のリポジトリ候補が見つかりました。
★ 10 件のデータを保存しました。


In [3]:
# スター数が多い順にトップ10を表示
cursor.execute("SELECT * FROM google_repos ORDER BY stars DESC LIMIT 10")
rows = cursor.fetchall()

print("--- 保存データ確認 (Top 10) ---")
print(f"{'Repository':<30} | {'Language':<15} | {'Stars'}")
print("-" * 60)
for row in rows:
    print(f"{row[0]:<30} | {row[1]:<15} | {row[2]}")

# 処理が終わったら接続を閉じる
conn.close()

--- 保存データ確認 (Top 10) ---
Repository                     | Language        | Stars
------------------------------------------------------------
